# Coupled fluxonia and normal modes

Two fluxonia [Manucharyan et al., Science **326**, 113 (2009)](https://doi.org/10.1126/science.1175552) share a static coupling capacitor $C_c$.
In node coordinates the capacitor couples their charges.
This example removes that quadratic coupling with normal-mode coordinates, first symbolically for identical fluxonia, then numerically for distinct ones.

In [ ]:
import numpy as np
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize

## Identical fluxonia, symbolically

Each fluxonium is a junction (with its capacitance $C$) in parallel with an inductor $L$; both loops are at zero flux.

In [ ]:
C, C_c, L, E_J = sym.symbols("C C_c L E_J", positive=True)

pair = Circuit([
    Branch("J1", 1, 0, C=C, EJ=E_J), Branch("L1", 1, 0, L=L),
    Branch("J2", 2, 0, C=C, EJ=E_J), Branch("L2", 2, 0, L=L),
    Branch("Cc", 1, 2, C=C_c),
], ground=0)
cq.draw_circuit_diagram(pair, layout={1: (0, 0), 2: (1.5, 0)}, grounds="below")

In [ ]:
branches = {"f1": ("J1", "L1"), "f2": ("J2", "L2"), "coupler": ("J1", "Cc", "J2")}
loops = Loops.from_branch_dict(pair, branches, {"f1": 0, "f2": 0, "coupler": 0})

result = quantize(pair, loops, simplify=True)
assert result.coords.types == ("e", "e")
result.ham.E_C

The off-diagonal entries of the charging-energy matrix are the capacitive coupling.
`normal_mode_decoupling` returns the transformation $S_1$ that diagonalizes both quadratic forms; `transform` applies it.

In [ ]:
S, *_ = cq.normal_mode_decoupling(result.lag, result.coords)
normal = result.transform(S, simplify=True)
for M in (normal.ham.E_C, normal.ham.E_L):
    assert sym.simplify(M[0, 1]) == 0 and sym.simplify(M[1, 0]) == 0
normal.coords.Z

Each column of `Z` gives one new coordinate in terms of the node phases.
The first is the symmetric mode and the second the antisymmetric mode, which alone charges the coupler:

In [ ]:
normal.ham.E_C, normal.ham.E_L

In [ ]:
normal

The quadratic part is now diagonal, but the junctions still couple the modes through $-2E_J\cos\varphi_1\cos\varphi_2$.

## Distinct fluxonia, numerically

For distinct parameters the symbolic eigenvectors are unwieldy, so the parameters are set to numbers.

In [ ]:
numeric = Circuit([
    Branch("J1", 1, 0, C=80e-15, EJ=4.0), Branch("L1", 1, 0, L=300e-9),
    Branch("J2", 2, 0, C=70e-15, EJ=4.5), Branch("L2", 2, 0, L=250e-9),
    Branch("Cc", 1, 2, C=10e-15),
], ground=0)
numeric_result = quantize(numeric, Loops.from_branch_dict(numeric, branches, {"f1": 0, "f2": 0, "coupler": 0}))

`numerical_S1` minimizes a cost over invertible transformations; `cost_normal_modes` is zero exactly at the normal modes.
With `values={}` every parameter is already numeric.

In [ ]:
S_search, info = cq.numerical_S1(numeric_result.lag, numeric_result.coords, values={},
                                 cost=cq.cost_normal_modes, relaxed=True)
assert info.cost < 1e-12
info.cost

The analytic route solves the same generalized eigenproblem directly.
Both give the same diagonal energies in GHz, up to the sign of each column:

In [ ]:
S_exact, *_ = cq.normal_mode_decoupling(numeric_result.lag, numeric_result.coords, values={})

def energies(S):
    ham = numeric_result.transform(S).ham
    return [np.array(cq.subs_constants(M), dtype=float) for M in (ham.E_C, ham.E_L)]

(E_C_search, E_L_search), (E_C_exact, E_L_exact) = energies(S_search), energies(S_exact)
assert np.allclose(np.diag(E_L_search), np.diag(E_L_exact), rtol=1e-6)
assert np.allclose(np.diag(E_C_search), np.diag(E_C_exact), rtol=1e-6)
E_C_search, E_L_search

The numerical search becomes useful when the target is not the normal modes: `cost_targeted` removes only selected couplings, and `cost_localization` favours coordinates that keep each junction on few modes.